# 01 - Data Exploration: SMS Spam Detection

## Overview
This notebook performs Exploratory Data Analysis (EDA) on the SMS Spam Collection dataset.

### Objectives:
1. Load and inspect the dataset
2. Analyze class distribution (spam vs ham)
3. Explore message length statistics
4. Visualize word frequencies
5. Generate WordClouds for spam and ham messages

---

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from wordcloud import WordCloud
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Set style for visualizations
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print("Libraries imported successfully!")

## 2. Load Dataset

The SMS Spam Collection dataset contains 5,574 SMS messages labeled as either 'spam' or 'ham' (legitimate).

**Dataset Source:** [UCI Machine Learning Repository](https://archive.ics.uci.edu/ml/datasets/SMS+Spam+Collection) or [Kaggle](https://www.kaggle.com/datasets/uciml/sms-spam-collection-dataset)

In [ ]:
# Load the dataset
# Try different encodings as the file may use Latin-1
try:
    df = pd.read_csv('../data/raw/spam.csv', encoding='utf-8')
except:
    df = pd.read_csv('../data/raw/spam.csv', encoding='latin-1')

# Display first few rows
print("Dataset loaded successfully!")
print(f"Shape: {df.shape}")
df.head()

In [ ]:
# Clean column names - keep only relevant columns
# The dataset typically has columns: v1 (label), v2 (message)
if 'v1' in df.columns and 'v2' in df.columns:
    df = df[['v1', 'v2']]
    df.columns = ['label', 'message']
elif 'Label' in df.columns:
    df.columns = ['label', 'message'] + list(df.columns[2:])
    df = df[['label', 'message']]

# Create binary label column
df['label_encoded'] = df['label'].map({'ham': 0, 'spam': 1})

print("Columns after cleaning:")
print(df.columns.tolist())
df.head()

## 3. Basic Dataset Information

In [ ]:
# Display dataset information
print("=" * 50)
print("DATASET INFORMATION")
print("=" * 50)
print(f"\nTotal Messages: {len(df)}")
print(f"Number of Features: {df.shape[1]}")
print(f"\nData Types:")
print(df.dtypes)
print(f"\nMissing Values:")
print(df.isnull().sum())

In [ ]:
# Statistical summary of the data
df.describe()

## 4. Class Distribution Analysis

Let's examine the distribution of spam vs ham messages to understand if we have a class imbalance.

In [ ]:
# Calculate class distribution
class_counts = df['label'].value_counts()
class_percentages = df['label'].value_counts(normalize=True) * 100

print("=" * 50)
print("CLASS DISTRIBUTION")
print("=" * 50)
print(f"\nHam (Legitimate) Messages: {class_counts['ham']} ({class_percentages['ham']:.2f}%)")
print(f"Spam Messages: {class_counts['spam']} ({class_percentages['spam']:.2f}%)")
print(f"\nImbalance Ratio: {class_counts['ham'] / class_counts['spam']:.2f}:1")

In [ ]:
# Visualize class distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar plot
colors = ['#2ecc71', '#e74c3c']
axes[0].bar(class_counts.index, class_counts.values, color=colors, edgecolor='black')
axes[0].set_xlabel('Message Type', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
axes[0].set_title('Class Distribution (Count)', fontsize=14)
for i, v in enumerate(class_counts.values):
    axes[0].text(i, v + 50, str(v), ha='center', fontsize=12, fontweight='bold')

# Pie chart
axes[1].pie(class_counts.values, labels=class_counts.index, autopct='%1.1f%%',
            colors=colors, explode=[0, 0.1], shadow=True, startangle=90)
axes[1].set_title('Class Distribution (Percentage)', fontsize=14)

plt.tight_layout()
plt.savefig('../results/plots/class_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nNote: The dataset is imbalanced with more ham messages than spam.")

## 5. Message Length Analysis

Analyzing message length can reveal patterns that distinguish spam from legitimate messages.

In [ ]:
# Add message length columns
df['message_length'] = df['message'].apply(len)
df['word_count'] = df['message'].apply(lambda x: len(str(x).split()))

# Statistics by class
print("=" * 60)
print("MESSAGE LENGTH STATISTICS")
print("=" * 60)

for label in ['ham', 'spam']:
    subset = df[df['label'] == label]
    print(f"\n{label.upper()} Messages:")
    print(f"  Average Length: {subset['message_length'].mean():.2f} characters")
    print(f"  Median Length: {subset['message_length'].median():.2f} characters")
    print(f"  Max Length: {subset['message_length'].max()} characters")
    print(f"  Min Length: {subset['message_length'].min()} characters")
    print(f"  Average Words: {subset['word_count'].mean():.2f} words")

In [ ]:
# Visualize message length distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Character length distribution
for label, color in zip(['ham', 'spam'], ['#2ecc71', '#e74c3c']):
    subset = df[df['label'] == label]['message_length']
    axes[0].hist(subset, bins=50, alpha=0.7, label=label, color=color, edgecolor='black')

axes[0].set_xlabel('Message Length (characters)', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].set_title('Distribution of Message Length', fontsize=14)
axes[0].legend()

# Word count distribution
for label, color in zip(['ham', 'spam'], ['#2ecc71', '#e74c3c']):
    subset = df[df['label'] == label]['word_count']
    axes[1].hist(subset, bins=50, alpha=0.7, label=label, color=color, edgecolor='black')

axes[1].set_xlabel('Word Count', fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)
axes[1].set_title('Distribution of Word Count', fontsize=14)
axes[1].legend()

plt.tight_layout()
plt.savefig('../results/plots/message_length_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Box plot comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Character length box plot
df.boxplot(column='message_length', by='label', ax=axes[0])
axes[0].set_xlabel('Message Type', fontsize=12)
axes[0].set_ylabel('Character Count', fontsize=12)
axes[0].set_title('Message Length by Type', fontsize=14)
plt.suptitle('')  # Remove automatic title

# Word count box plot
df.boxplot(column='word_count', by='label', ax=axes[1])
axes[1].set_xlabel('Message Type', fontsize=12)
axes[1].set_ylabel('Word Count', fontsize=12)
axes[1].set_title('Word Count by Type', fontsize=14)
plt.suptitle('')

plt.tight_layout()
plt.savefig('../results/plots/message_length_boxplot.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nObservation: Spam messages tend to be longer than ham messages on average.")

## 6. Word Frequency Analysis

In [ ]:
# Function to get most common words
def get_top_words(texts, n=20):
    """Extract top n most common words from a collection of texts."""
    all_words = []
    for text in texts:
        words = str(text).lower().split()
        all_words.extend(words)
    return Counter(all_words).most_common(n)

# Get top words for ham and spam
ham_words = get_top_words(df[df['label'] == 'ham']['message'])
spam_words = get_top_words(df[df['label'] == 'spam']['message'])

print("TOP 10 WORDS IN HAM MESSAGES:")
for word, count in ham_words[:10]:
    print(f"  {word}: {count}")

print("\nTOP 10 WORDS IN SPAM MESSAGES:")
for word, count in spam_words[:10]:
    print(f"  {word}: {count}")

In [ ]:
# Visualize top words
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Ham words
words, counts = zip(*ham_words[:15])
axes[0].barh(words, counts, color='#2ecc71', edgecolor='black')
axes[0].set_xlabel('Frequency', fontsize=12)
axes[0].set_title('Top 15 Words in HAM Messages', fontsize=14)
axes[0].invert_yaxis()

# Spam words
words, counts = zip(*spam_words[:15])
axes[1].barh(words, counts, color='#e74c3c', edgecolor='black')
axes[1].set_xlabel('Frequency', fontsize=12)
axes[1].set_title('Top 15 Words in SPAM Messages', fontsize=14)
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig('../results/plots/top_words.png', dpi=300, bbox_inches='tight')
plt.show()

## 7. Word Cloud Visualization

In [ ]:
# Generate word clouds
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Ham word cloud
ham_text = ' '.join(df[df['label'] == 'ham']['message'].astype(str))
ham_wordcloud = WordCloud(width=800, height=400, background_color='white',
                          colormap='Greens', max_words=100).generate(ham_text)
axes[0].imshow(ham_wordcloud, interpolation='bilinear')
axes[0].axis('off')
axes[0].set_title('Word Cloud: HAM Messages', fontsize=16)

# Spam word cloud
spam_text = ' '.join(df[df['label'] == 'spam']['message'].astype(str))
spam_wordcloud = WordCloud(width=800, height=400, background_color='white',
                           colormap='Reds', max_words=100).generate(spam_text)
axes[1].imshow(spam_wordcloud, interpolation='bilinear')
axes[1].axis('off')
axes[1].set_title('Word Cloud: SPAM Messages', fontsize=16)

plt.tight_layout()
plt.savefig('../results/plots/wordclouds.png', dpi=300, bbox_inches='tight')
plt.show()

## 8. Sample Messages

In [ ]:
# Display sample ham messages
print("=" * 60)
print("SAMPLE HAM (LEGITIMATE) MESSAGES")
print("=" * 60)
for i, msg in enumerate(df[df['label'] == 'ham']['message'].head(5)):
    print(f"\n{i+1}. {msg}")

In [ ]:
# Display sample spam messages
print("=" * 60)
print("SAMPLE SPAM MESSAGES")
print("=" * 60)
for i, msg in enumerate(df[df['label'] == 'spam']['message'].head(5)):
    print(f"\n{i+1}. {msg}")

## 9. Key Findings Summary

### Dataset Characteristics:
- **Total Messages:** 5,574
- **Class Imbalance:** ~87% ham, ~13% spam
- **Imbalance Ratio:** Approximately 6-7:1 (ham:spam)

### Message Length Patterns:
- Spam messages are generally **longer** than ham messages
- Spam messages contain more words on average
- Ham messages vary more in length

### Word Patterns:
- Spam messages contain words like: "free", "call", "txt", "win", "prize"
- Ham messages contain more conversational words

### Implications for Modeling:
1. Need to handle class imbalance (stratified splitting, class weights)
2. Message length could be a useful feature
3. Certain keywords are strong spam indicators

---
**Next Step:** Proceed to `02_preprocessing.ipynb` for data preprocessing.

In [ ]:
# Save the analyzed data for next notebook
df.to_csv('../data/processed/analyzed_data.csv', index=False)
print("Analyzed data saved to '../data/processed/analyzed_data.csv'")